# Session 4: Exception Handling & Debugging

**Course:** Python for Data Engineering  
**Phase 1:** Foundations

**What we'll cover:**
- try-except-finally
- Common errors in data pipelines
- Raising exceptions
- Debugging techniques
- Lab: Build fault-tolerant pipelines

**Data files:** We'll use files in the `data/` directory — same data you've been working with in Sessions 1–3 (sales, employees, transactions), plus `data/sales_messy.csv` which has intentionally bad data.

---

## 1. Why Exception Handling Matters

In production pipelines, things go wrong constantly:
- A CSV has a blank field where you expected a number
- An API returns a timeout instead of data
- A file path doesn't exist
- A JSON response has an unexpected structure

Without exception handling, **one bad row kills the entire pipeline**. With it, you can skip bad records, log the issue, and keep going.

### The basic pattern

```python
try:
    # code that might fail
except SomeError:
    # what to do when it fails
```

---

## 2. Common Errors in Data Pipelines

Before we handle errors, let's see what they look like. These are the ones you'll hit most often.

| Error | What causes it | Pipeline example |
|-------|---------------|------------------|
| `ValueError` | Wrong value for conversion | `int("abc")`, `float("free")` |
| `KeyError` | Missing key in dict | `row["price"]` when column is `"unit_price"` |
| `TypeError` | Wrong type in operation | `"10" + 5` (str + int) |
| `FileNotFoundError` | File doesn't exist | `open("data/missing.csv")` |
| `IndexError` | Index out of range | `parts[5]` on a 3-element list |
| `ZeroDivisionError` | Dividing by zero | `total / count` when count is 0 |

In [ ]:
# ValueError — the most common one in data pipelines
# happens when CSV has bad data in a numeric column

raw_quantity = "ten"  # should be "10" but someone typed the word

# This will crash
# num = int(raw_quantity)  # ValueError: invalid literal for int()

# Same thing with prices
raw_price = "free"
# num = float(raw_price)  # ValueError

In [ ]:
# KeyError — column name mismatch
# happens when source data changes schema

record = {"product": "Laptop", "unit_price": 999.99, "qty": 2}

# This works
print(record["product"])

# This crashes — column is "qty" not "quantity"
# print(record["quantity"])  # KeyError: 'quantity'

In [ ]:
# TypeError — mixing types

quantity = "10"  # string from CSV
price = 29.99    # float

# This crashes — can't multiply string by float
# total = quantity * price  # TypeError

# Fix: convert first
total = int(quantity) * price
print(f"Total: ${total:.2f}")

In [ ]:
# FileNotFoundError — missing source file

# This crashes — file doesn't exist
# with open("data/does_not_exist.csv") as f:
#     data = f.read()  # FileNotFoundError

# IndexError — parsing a line with fewer fields than expected
log_line = "2024-01-15 | INFO | user_login"  # missing user_id and ip
parts = log_line.split(" | ")
print(f"Parts: {parts}, Length: {len(parts)}")

# This crashes — only 3 elements, no index 3
# user_id = parts[3]  # IndexError

**Try it:** Run each of the commented-out lines above (one at a time) to see the actual error messages. Read the traceback — it tells you the exact line and error type.

In [ ]:
# Your code here — uncomment and run the lines to see errors

---

## 3. try-except

Wrap risky code in `try`, handle the failure in `except`. The program keeps running.

In [ ]:
# Basic try-except — handle bad quantity from CSV

raw_values = ["10", "5", "abc", "3", "", "7"]

clean = []
bad = []

for val in raw_values:
    try:
        clean.append(int(val))
    except ValueError:
        bad.append(val)

print(f"Clean: {clean}")
print(f"Bad:   {bad}")

In [ ]:
# Catching multiple error types

records = [
    {"product": "Laptop", "quantity": "2", "unit_price": "$999.99"},
    {"product": "Mouse", "quantity": "ten"},                         # missing unit_price + bad qty
    {"product": "Keyboard", "quantity": "5", "unit_price": "$79.50"},
    {"product": "Monitor", "quantity": "1", "unit_price": "free"},    # bad price
]

processed = []
errors = []

for row in records:
    try:
        qty = int(row["quantity"])
        price = float(row["unit_price"].replace("$", ""))
        processed.append({
            "product": row["product"],
            "quantity": qty,
            "total": round(qty * price, 2)
        })
    except ValueError as e:
        errors.append({"product": row["product"], "error": f"Bad value: {e}"})
    except KeyError as e:
        errors.append({"product": row["product"], "error": f"Missing field: {e}"})

print(f"Processed: {len(processed)}, Errors: {len(errors)}")
for r in processed:
    print(f"  {r['product']:12s} qty={r['quantity']} total=${r['total']:>10,.2f}")
print("\nErrors:")
for e in errors:
    print(f"  {e['product']:12s} — {e['error']}")

In [ ]:
# Using the error object — the `as e` part

try:
    value = int("hello")
except ValueError as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")
    # in a pipeline, you'd log this and continue

**Try it:** Process these salary strings. Convert each to a float (remove `$` and `,`). Catch errors for bad values, collect them separately.

```python
salaries = ["$65,000.00", "$82,500.50", "not_a_salary", "$95,000.00", "", "$88,750.25"]
```

In [ ]:
salaries = ["$65,000.00", "$82,500.50", "not_a_salary", "$95,000.00", "", "$88,750.25"]

# Your code here

---

## 4. try-except-else-finally

The full pattern:

```python
try:
    # risky code
except SomeError:
    # handle failure
else:
    # runs only if try succeeded (no error)
finally:
    # always runs — cleanup code
```

| Block | When it runs |
|-------|-------------|
| `try` | Always (the code you're protecting) |
| `except` | Only if an error occurs |
| `else` | Only if **no** error occurs |
| `finally` | **Always** — whether error or not |

In [ ]:
# Reading a file with full error handling

import csv

filename = "data/sales.csv"

try:
    f = open(filename, "r")
    reader = csv.DictReader(f)
    rows = list(reader)
except FileNotFoundError:
    print(f"File not found: {filename}")
    rows = []
else:
    # only runs if file was read successfully
    print(f"Loaded {len(rows)} rows from {filename}")
finally:
    # cleanup — close the file if it was opened
    if 'f' in dir() and not f.closed:
        f.close()
        print("File closed")

In [ ]:
# Now try with a file that doesn't exist

filename = "data/does_not_exist.csv"

try:
    f = open(filename, "r")
    reader = csv.DictReader(f)
    rows = list(reader)
except FileNotFoundError:
    print(f"File not found: {filename}")
    rows = []
else:
    print(f"Loaded {len(rows)} rows")
finally:
    print("Cleanup done")

print(f"Rows available: {len(rows)}")

In [ ]:
# finally is most useful for cleanup — closing connections, temp files, etc.

import json

def load_config(path):
    """Load pipeline config. Returns empty dict if anything goes wrong."""
    try:
        with open(path, "r") as f:
            config = json.load(f)
    except FileNotFoundError:
        print(f"Config not found: {path}")
        return {}
    except json.JSONDecodeError as e:
        print(f"Invalid JSON in {path}: {e}")
        return {}
    else:
        print(f"Config loaded from {path}")
        return config

# Test with good config
config = load_config("data/pipeline_config.json")
print(f"Pipeline: {config.get('pipeline_name', 'unknown')}")

# Test with missing file
config = load_config("data/nope.json")
print(f"Pipeline: {config.get('pipeline_name', 'unknown')}")

**Try it:** Write a function `safe_read_csv(filepath)` that:
1. Tries to open and read a CSV file with `csv.DictReader`
2. Returns the list of rows if successful
3. Returns an empty list if `FileNotFoundError`
4. Prints a message either way

Test with `data/sales.csv` (exists) and `data/orders.csv` (doesn't exist).

In [ ]:
import csv

# Your code here

---

## 5. Raising Exceptions

Sometimes **you** want to raise an error — when data doesn't meet your pipeline's requirements.

```python
raise ValueError("quantity must be positive")
```

This is how you enforce data quality rules.

In [ ]:
# Validating a sales record before processing

def validate_sale(record):
    """Check that a sales record has all required fields and valid values."""
    required_fields = ["product", "quantity", "unit_price", "region"]
    
    for field in required_fields:
        if field not in record:
            raise KeyError(f"Missing required field: {field}")
        if not str(record[field]).strip():
            raise ValueError(f"Field '{field}' is empty")
    
    if record["quantity"] <= 0:
        raise ValueError(f"Quantity must be positive, got {record['quantity']}")
    
    if record["unit_price"] < 0:
        raise ValueError(f"Price cannot be negative, got {record['unit_price']}")

# Good record
try:
    validate_sale({"product": "Laptop", "quantity": 2, "unit_price": 999.99, "region": "north"})
    print("Record is valid")
except (KeyError, ValueError) as e:
    print(f"Validation failed: {e}")

# Bad record — negative quantity
try:
    validate_sale({"product": "Monitor", "quantity": -1, "unit_price": 349.99, "region": "east"})
    print("Record is valid")
except (KeyError, ValueError) as e:
    print(f"Validation failed: {e}")

# Bad record — missing field
try:
    validate_sale({"product": "Mouse", "quantity": 5})
    print("Record is valid")
except (KeyError, ValueError) as e:
    print(f"Validation failed: {e}")

In [ ]:
# Using validation in a pipeline — process only valid records

sales_batch = [
    {"product": "Laptop", "quantity": 2, "unit_price": 999.99, "region": "north"},
    {"product": "Mouse", "quantity": -3, "unit_price": 29.99, "region": "south"},
    {"product": "Keyboard", "quantity": 5, "unit_price": 79.50, "region": "north"},
    {"product": "", "quantity": 1, "unit_price": 349.99, "region": "east"},
    {"product": "Headphones", "quantity": 3, "unit_price": 59.99, "region": "north"},
]

valid = []
rejected = []

for sale in sales_batch:
    try:
        validate_sale(sale)
        total = round(sale["quantity"] * sale["unit_price"], 2)
        sale["total"] = total
        valid.append(sale)
    except (KeyError, ValueError) as e:
        rejected.append({"record": sale, "reason": str(e)})

print(f"Valid: {len(valid)}, Rejected: {len(rejected)}")
print("\nValid records:")
for s in valid:
    print(f"  {s['product']:12s} qty={s['quantity']} total=${s['total']:>10,.2f}")
print("\nRejected:")
for r in rejected:
    print(f"  {r['record'].get('product', '???'):12s} — {r['reason']}")

**Try it:** Write a `validate_employee(record)` function that checks:
- Has `name`, `age`, `salary`, `department` fields
- Name is not empty
- Age is between 18 and 65
- Salary is positive
- Department is one of: `engineering`, `data`, `hr`

Test with a few good and bad employee records.

In [ ]:
# Your code here

---

## 6. Handling Bad Data from Files

Let's put it all together with real file data. `data/sales_messy.csv` has intentionally bad records — missing products, non-numeric quantities, invalid prices.

In [ ]:
# First, let's see what the messy data looks like

with open("data/sales_messy.csv", "r") as f:
    print(f.read())

In [ ]:
# Processing messy data with error handling — row by row

import csv

clean_records = []
error_records = []

with open("data/sales_messy.csv", "r") as f:
    reader = csv.DictReader(f)
    for row_num, row in enumerate(reader, start=2):  # start=2 because row 1 is header
        try:
            # validate product name
            product = row["product"].strip()
            if not product:
                raise ValueError("empty product name")
            
            # convert quantity
            quantity = int(row["quantity"])
            if quantity <= 0:
                raise ValueError(f"invalid quantity: {quantity}")
            
            # convert price
            price_str = row["unit_price"].replace("$", "").strip()
            price = float(price_str)
            
            # validate region
            region = row["region"].strip().lower()
            if not region:
                raise ValueError("empty region")
            
            clean_records.append({
                "date": row["date"],
                "product": product.title(),
                "quantity": quantity,
                "unit_price": price,
                "total": round(quantity * price, 2),
                "region": region,
            })
        except (ValueError, KeyError) as e:
            error_records.append({
                "row": row_num,
                "data": dict(row),
                "error": str(e),
            })

print(f"Clean: {len(clean_records)}, Errors: {len(error_records)}")
print("\nClean records:")
for r in clean_records:
    print(f"  {r['product']:12s} qty={r['quantity']} ${r['total']:>10,.2f} ({r['region']})")

print("\nError records:")
for e in error_records:
    print(f"  Row {e['row']}: {e['error']}")

In [ ]:
# Writing error records to a separate file for review

import json

with open("data/sales_errors.json", "w") as f:
    json.dump(error_records, f, indent=2)

print(f"Wrote {len(error_records)} error records to data/sales_errors.json")

# Verify
with open("data/sales_errors.json", "r") as f:
    print(f.read())

**Try it:** Read `data/employees.csv` with error handling:
1. Try to convert age to int and salary to float (remove `$`)
2. Catch any ValueError for bad conversions
3. Skip rows where department is `"unknown"` (raise your own ValueError for this)
4. Print clean records and error records separately

In [ ]:
import csv

# Your code here

---

## 7. Debugging Techniques

When something goes wrong (and it will), here's how to find the problem.

### Technique 1: Read the traceback

Python's error messages are actually very helpful — read them **bottom to top**.

```
Traceback (most recent call last):
  File "pipeline.py", line 25, in process_row
    price = float(row["unit_price"].replace("$", ""))
ValueError: could not convert string to float: 'free'
```

This tells you: **what** happened (`ValueError`), **where** (line 25, `process_row`), and **why** (tried to convert `'free'` to float).

In [ ]:
# Technique 2: Strategic print statements
# Add prints before the line that fails to see what the data looks like

records = [
    {"product": "Laptop", "quantity": "2", "unit_price": "$999.99"},
    {"product": "Mouse", "quantity": "abc", "unit_price": "$29.99"},
]

for i, row in enumerate(records):
    print(f"DEBUG row {i}: quantity='{row['quantity']}', price='{row['unit_price']}'")
    try:
        qty = int(row["quantity"])
        price = float(row["unit_price"].replace("$", ""))
        print(f"DEBUG row {i}: converted qty={qty}, price={price}")
    except ValueError as e:
        print(f"DEBUG row {i}: FAILED — {e}")

In [ ]:
# Technique 3: Type checking — when you're not sure what you're working with

mystery_data = [42, "hello", 3.14, None, True, [1, 2]]

for item in mystery_data:
    print(f"Value: {str(item):10s} Type: {type(item).__name__:6s} Bool: {bool(item)}")

In [ ]:
# Technique 4: assert — quick sanity checks during development
# these get removed in production but are great for catching bugs early

def process_batch(records):
    assert isinstance(records, list), f"Expected list, got {type(records)}"
    assert len(records) > 0, "Empty batch — nothing to process"
    
    print(f"Processing {len(records)} records...")
    # ... rest of processing

# This works
process_batch([{"product": "Laptop"}, {"product": "Mouse"}])

# This fails with a clear message
try:
    process_batch([])
except AssertionError as e:
    print(f"Assertion failed: {e}")

try:
    process_batch("not a list")
except AssertionError as e:
    print(f"Assertion failed: {e}")

In [ ]:
# Technique 5: Logging with context — from Session 3, now with error details

import logging

log = logging.getLogger("debug_demo")
log.setLevel(logging.DEBUG)
log.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
log.addHandler(handler)

# Simulating a pipeline that processes transactions
raw_prices = ["$999.99", "$29.99", "free", "$79.50", None, "$59.99"]

clean_prices = []
for i, price in enumerate(raw_prices):
    try:
        if price is None:
            raise TypeError("price is None")
        clean = float(price.replace("$", ""))
        clean_prices.append(clean)
        log.debug(f"Row {i}: '{price}' -> {clean}")
    except (ValueError, TypeError) as e:
        log.warning(f"Row {i}: skipped — {e}")

log.info(f"Processed {len(clean_prices)}/{len(raw_prices)} prices")
log.info(f"Total: ${sum(clean_prices):,.2f}")

**Try it:** You have this buggy code. Find and fix the bugs using the techniques above.

```python
data = [
    {"name": "Alice Johnson", "age": "32", "salary": "$75,000"},
    {"name": "Bob Smith", "age": "twenty-eight", "salary": "$65,000"},
    {"name": "Charlie Brown", "age": "45", "salary": "$90,000"},
]

total_salary = 0
for emp in data:
    age = int(emp["age"])
    salary = float(emp["salary"])
    total_salary += salary
```

The code crashes. Add error handling so it processes what it can and skips what it can't.

In [ ]:
# Your code here

---

## Lab Exercises

---

### Lab 1: Handle Bad Data Inputs

Read `data/sales_messy.csv` and build a fault-tolerant processing pipeline.

**Steps:**
1. Read each row with `csv.DictReader`
2. For each row, try to:
   - Convert quantity to int (reject if not a number)
   - Convert unit_price to float (remove `$`, reject if not a number)
   - Check product is not empty
   - Check region is not empty
   - Check quantity is positive
3. Collect clean records and error records separately
4. Write clean records to `data/sales_validated.csv`
5. Write error records to `data/sales_rejected.json` with row number and error reason
6. Print a summary: total clean, total rejected, total revenue from clean records

In [ ]:
import csv
import json

# Your code here

---

### Lab 2: Fault-Tolerant ETL Pipeline

Build a complete pipeline that handles every possible failure. Use `data/sales.csv` and `data/transactions.json`.

**Steps:**
1. Try to load config from `data/pipeline_config.json` — use defaults if it fails
2. Try to read `data/sales.csv` — if missing, log error and use empty list
3. Try to read `data/transactions.json` — if missing, log error and use empty list
4. Process each sales row with try-except (same cleaning as Session 3)
5. Process each transaction with try-except
6. Combine results into a single summary
7. Log every step with proper logging (INFO for progress, WARNING for skipped, ERROR for failures)
8. Write summary to `data/pipeline_report.json` with:
   - `sales_processed`, `sales_errors`
   - `transactions_processed`, `transactions_errors`
   - `total_revenue`
   - `status` ("success" if no errors, "partial" if some errors, "failed" if all errors)

In [ ]:
import csv
import json
import logging
from datetime import datetime

# Your code here

---

### Lab 3: Build a Retry Wrapper

In real pipelines, things fail temporarily — a file might be locked, a network call times out. A common pattern is to **retry** before giving up.

**Steps:**
1. Write a function `retry(func, max_attempts=3)` that:
   - Calls `func()`
   - If it raises an exception, prints the error and tries again
   - Gives up after `max_attempts` and re-raises the last error
   - Returns the result if successful
2. Test it with a function that fails randomly:

```python
import random

def unreliable_fetch():
    if random.random() < 0.6:  # 60% chance of failure
        raise ConnectionError("API timeout")
    return {"status": "ok", "data": [1, 2, 3]}
```

3. Add logging to track each attempt
4. Bonus: add a `delay` parameter that waits between retries (use `import time; time.sleep(delay)`)

In [ ]:
import random
import time

# Your code here

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| Common errors | `ValueError`, `KeyError`, `TypeError`, `FileNotFoundError` — know what causes them |
| try-except | Catch errors and keep the pipeline running |
| Multiple except | Handle different error types differently |
| else/finally | `else` runs on success, `finally` always runs (cleanup) |
| raise | Enforce your own validation rules |
| Debugging | Read tracebacks, add prints, check types, use assert, log with context |

**Key patterns:**
- Never let one bad row kill the whole pipeline
- Catch specific errors, not generic `except:` (that hides bugs)
- Collect errors separately — write them to a file for review
- Validate early, fail fast — check data before processing
- Use `raise` to enforce your data quality rules
- Log the error + the data that caused it — you'll need both to debug

**This wraps up Phase 1: Foundations.** You now know Python basics, control flow, file handling, logging, and error handling — everything you need to start building real pipelines.

**Next session:** Working with Libraries — virtual environments, pip, and setting up pandas + numpy.